# Project 3: MediaPipe Gesture Recognition

Course lesson: [P3.2 The MediaPipe Colab Notebook](https://neel429.github.io/ml-for-robotics-/index.html?lesson=proj3-mediapipe-colab)

This notebook teaches MediaPipe hand tracking on a still image before you run the live webcam test in VS Code and connect gestures to the robot.

## Cell 1: Install MediaPipe and download the model

The current MediaPipe Tasks API separates the Python library from the trained model weights. This cell installs the library and downloads `hand_landmarker.task`, the model bundle used by the hand landmark detector.

In [ ]:
!pip install -q mediapipe
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

import os
if os.path.exists("hand_landmarker.task"):
    size_mb = os.path.getsize("hand_landmarker.task") / 1024 / 1024
    print(f"Model downloaded successfully ({size_mb:.1f} MB)")
else:
    print("ERROR: Model file not found. Re-run this cell.")

> **Why two steps to load the model?** Cell 1 downloads the model weights file, about 8MB, from Google's servers and saves it as `hand_landmarker.task` in the Colab session's local storage. Cell 2 loads that file into memory and initializes the detector. Both cells are required. The `-q` flag in `wget` makes the download silent; it gives no output even when it succeeds. The verification print above confirms it worked. If the file size shows as 0 MB or the file is missing, your internet connection blocked the download. Try re-running Cell 1.

## Cell 2: Initialize the HandLandmarker

`RunningMode.IMAGE` is for still images. The VS Code files use `RunningMode.VIDEO` because they process camera frames with timestamps.

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (5, 6), (6, 7), (7, 8),
    (5, 9), (9, 10), (10, 11), (11, 12),
    (9, 13), (13, 14), (14, 15), (15, 16),
    (13, 17), (0, 17), (17, 18), (18, 19), (19, 20),
]

def draw_hand_landmarks(rgb_image, hand_landmarks_list):
    annotated = np.copy(rgb_image)
    h, w, _ = annotated.shape

    for hand_landmarks in hand_landmarks_list:
        pts = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
        for start, end in HAND_CONNECTIONS:
            cv2.line(annotated, pts[start], pts[end], (0, 255, 0), 2)
        for pt in pts:
            cv2.circle(annotated, pt, 4, (0, 0, 255), -1)

    return annotated

base_options = mp_python.BaseOptions(model_asset_path="hand_landmarker.task")
options = mp_vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.7,
)
landmarker = mp_vision.HandLandmarker.create_from_options(options)
print("HandLandmarker ready.")

## Cell 3: Test on a still image

Upload a photo of your own hand. The detector returns `result.hand_landmarks`, a list of hands, and `result.handedness`, the left/right label for each hand.

In [ ]:
uploaded = files.upload()   # upload your hand photo
image_path = list(uploaded.keys())[0]

mp_image = mp.Image.create_from_file(image_path)
result   = landmarker.detect(mp_image)

print(f"Hands detected: {len(result.hand_landmarks)}")

if result.hand_landmarks:
    for i, hand in enumerate(result.hand_landmarks):
        label = result.handedness[i][0].category_name
        print(f"\nHand {i+1}: {label}")
        for j in range(5):
            lm = hand[j]
            print(f"  Landmark {j}: x={lm.x:.3f}  y={lm.y:.3f}  z={lm.z:.3f}")

annotated = draw_hand_landmarks(mp_image.numpy_view(), result.hand_landmarks)
plt.figure(figsize=(8, 6))
plt.imshow(annotated)
plt.title("MediaPipe HandLandmarker result")
plt.axis("off")
plt.show()

## Cell 4: Read specific landmarks

A hand is now a direct list of 21 normalized landmarks. Use `result.hand_landmarks[0][8].y` to read the index fingertip y coordinate.

In [ ]:
if result.hand_landmarks:
    hand = result.hand_landmarks[0]

    print("Key landmarks for finger detection:")
    print(f"  Wrist (0):           x={hand[0].x:.3f}  y={hand[0].y:.3f}")
    print(f"  Index tip (8):       x={hand[8].x:.3f}  y={hand[8].y:.3f}")
    print(f"  Index PIP (6):       x={hand[6].x:.3f}  y={hand[6].y:.3f}")
    print()

    index_extended = hand[8].y < hand[6].y
    print(f"Index finger extended: {index_extended}")
    print(f"  tip.y={hand[8].y:.3f}  pip.y={hand[6].y:.3f}")
    print(f"  tip is {'ABOVE' if index_extended else 'BELOW'} PIP = finger is {'extended' if index_extended else 'curled'}")

## Cell 5: The is_hand_open function

The robot code uses this same landmark-list access pattern.

In [ ]:
def is_hand_open(hand_landmarks):
    pts = hand_landmarks
    count = 0

    if pts[4].x > pts[2].x:
        count += 1

    for tip, pip in zip([8, 12, 16, 20], [6, 10, 14, 18]):
        if pts[tip].y < pts[pip].y:
            count += 1

    return count >= 3

print("Testing is_hand_open on your uploaded photo:")
if result.hand_landmarks:
    open_state = is_hand_open(result.hand_landmarks[0])
    print(f"  Hand is: {'OPEN' if open_state else 'CLOSED'}")

## Cell 6: Two-hand gesture classification

In [ ]:
def classify_gesture(hands_dict):
    if 'Left' not in hands_dict or 'Right' not in hands_dict:
        return "stop"

    lo = hands_dict['Left']['open']
    ro = hands_dict['Right']['open']

    if lo and ro:
        return "forward"
    if not lo and not ro:
        return "backward"
    if ro and not lo:
        return "right"
    if lo and not ro:
        return "left"
    return "stop"

hands_dict = {}
for i, handedness_list in enumerate(result.handedness):
    label = handedness_list[0].category_name
    hands_dict[label] = {'open': is_hand_open(result.hand_landmarks[i])}

print("Detected hands:", hands_dict)
print("Command:", classify_gesture(hands_dict).upper())

| Left hand | Right hand | Command |
|---|---|---|
| OPEN | OPEN | FORWARD |
| CLOSED | CLOSED | BACKWARD |
| CLOSED | OPEN | TURN RIGHT |
| OPEN | CLOSED | TURN LEFT |
| One hand | --- | STOP |

Requiring both hands is a safety design: the robot only moves when you deliberately hold both hands in the frame.

## Cell 7: Live webcam test runs in VS Code, not Colab

Colab cannot access the laptop webcam directly in a standard cell. The `gesture_recognize.py` file in the repository runs live gesture detection on your laptop webcam with `RunningMode.VIDEO` and `detect_for_video()`.

The notebook teaches the concepts on still images. The live test runs locally. This is the same split as the lane follower project: understand the algorithm in Colab, apply it in VS Code.